[◀ Notebook 12: Iterators & Generators](12_iterators_and_generators.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 14: Metaprogramming ▶](14_metaprogramming.ipynb)

# Notebook 13: Context Managers and Decorators

Context Managers and Decorators are powerful syntactic tools in Python for wrapping code blocks and functions. They encapsulate setup/teardown mechanics, exception handling, and runtime behavior extensions.

### Python Language Reference & PEP Mapping:
- **PEP 343 (The 'with' Statement)**: Introduces context managers and the `with` block.
- **PEP 318 (Decorators for Functions and Methods)**: Introduces the `@decorator` syntax for functions.
- **PEP 3129 (Class Decorators)**: Extends decorator syntax to classes.

## 1. The Context Manager Protocol

A **context manager** is an object that defines the runtime context to be established when executing a `with` statement. The protocol consists of two magic methods:

1. **`__enter__(self)`**: Runs at the start of the block. Its return value is bound to the target variable in the `as` clause (if any).
2. **`__exit__(self, exc_type, exc_val, exc_tb)`**: Runs at the end of the block, regardless of whether the block exited normally or raised an exception.

### CPython Execution Flow of a `with` Statement:
When CPython encounters a `with context_expr as var:` block:
1. CPython evaluates `context_expr` to obtain a context manager object.
2. The manager's `__enter__()` method is loaded and called.
3. If an `as var` clause is present, the return value of `__enter__()` is assigned to `var`.
4. The code block inside the `with` statement is executed.
5. If an exception occurs during execution, or if the block finishes normally, the manager's `__exit__()` method is called. If an exception caused the exit, its type, value, and traceback are passed as `exc_type`, `exc_val`, and `exc_tb` respectively. Otherwise, all three arguments are `None`.

### Exception Handling and Suppression inside `__exit__`:
- If an exception is passed to `__exit__`, and the method returns a **truthy** value (such as `True`), CPython suppresses the exception and continues execution normally with the code following the `with` block.
- If `__exit__` returns `None` or any **falsy** value, the exception is re-raised immediately after the `__exit__` method finishes executing.

In [ ]:
class FileManager:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode
        self.file = None

    def __enter__(self):
        print('-> Entering __enter__')
        self.file = open(self.filename, self.mode)
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        print('-> Entering __exit__')
        if self.file:
            self.file.close()
        if exc_type is not None:
            print(f'-> Exception caught: {exc_val}')
            # Suppress ValueError exceptions by returning True
            if issubclass(exc_type, ValueError):
                print('-> Suppressing ValueError!')
                return True
        # Return False to let other exceptions propagate
        return False

# 1. Normal execution flow
print("--- Scenario A: Normal Exit ---")
with FileManager('temp.txt', 'w') as f:
    f.write('Hello Context Manager!')

import os
os.remove('temp.txt')

# 2. Exception suppression flow
print("\n--- Scenario B: Exception Suppression (ValueError) ---")
with FileManager('temp.txt', 'w') as f:
    raise ValueError("Something went wrong, but will be suppressed")

if os.path.exists('temp.txt'):
    os.remove('temp.txt')


### The `@contextmanager` Utility
Writing a class is not the only way to create a context manager. Python's `contextlib` module provides a `@contextmanager` decorator that wraps a generator function. The code before the `yield` statement acts as `__enter__`, and the code after behaves as `__exit__`. Exceptions raised inside the block are thrown back into the generator at the `yield` point, so exception handling must be managed using a `try-finally` block.

In [ ]:
from contextlib import contextmanager

@contextmanager
def temporary_directory_simulator(name):
    print(f'-> Creating directory: {name}')
    try:
        yield name
    finally:
        print(f'-> Deleting directory: {name}')

with temporary_directory_simulator('/tmp/my_dir') as path:
    print(f'Doing operations in {path}')

## 2. Decorators

A **decorator** is a callable that takes another function or class as an argument, extends or modifies its behavior, and returns a new callable. Function decorators run at **import time** (when the function is defined), wrapping the original function.

### Closure & Syntax Sugar
The `@decorator` syntax is syntactic sugar. The following code:
```python
@my_decorator
def my_func():
    pass
```
Is exactly equivalent to:
```python
my_func = my_decorator(my_func)
```

### Nested Functions and Lexical Closures
A decorator utilizes nested functions to define a wrapper that runs code before and after calling the original function. The wrapper function forms a *closure* over the `func` parameter of the outer decorator function, capturing and retaining access to it even after the outer decorator function completes and returns.

In [ ]:
def simple_logger(func):
    def wrapper(*args, **kwargs):
        print(f'Calling {func.__name__} with args: {args}, kwargs: {kwargs}')
        result = func(*args, **kwargs)
        print(f'{func.__name__} returned {result}')
        return result
    return wrapper

@simple_logger
def add(a, b):
    return a + b

add(3, 4)

### Metadata Preservation via `functools.wraps`
When you decorate a function, you are replacing it with the inner wrapper function. Consequently, standard metadata like the function's name (`__name__`), docstring (`__doc__`), and annotations are lost. To fix this, Python provides `functools.wraps`, which copies this metadata back onto the wrapper function.

In [ ]:
import functools

def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def good_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def test_bad():
    """This is test_bad's docstring"""
    pass

@good_decorator
def test_good():
    """This is test_good's docstring"""
    pass

print('Bad decorator name:', test_bad.__name__, '| doc:', test_bad.__doc__)
print('Good decorator name:', test_good.__name__, '| doc:', test_good.__doc__)

### Parameterized Decorators (Decorator Factories)
To pass arguments to a decorator, you must create a **decorator factory**—a function that accepts the parameters, and returns the actual decorator, which in turn accepts the target function and returns the wrapper. 

This structure results in three levels of nested functions:
1. **Outer Factory**: Receives the configuration arguments (e.g., `num_times`) and returns the decorator.
2. **Middle Decorator**: Receives the target function (`func`) and returns the wrapper.
3. **Inner Wrapper**: Receives the actual runtime arguments (`*args, **kwargs`) and runs the custom logic using the references captured in its lexical closures.

In [ ]:
def repeat(num_times):
    def decorator_repeat(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator_repeat

@repeat(num_times=3)
def greet(name):
    print(f'Hello {name}')

greet('Alice')

---
## Coding Challenges

Complete the following exercises to test your mastery of context managers and decorators.

### Challenge 1: Transaction Context Manager with Rollback

Implement a context manager `Transaction` that handles database transactions. You are given a mock database connection class `MockConnection`.

The context manager should:
1. Call `conn.begin()` when entering the block.
2. Yield/return the connection object.
3. Call `conn.commit()` when exiting, if no exception occurred.
4. Call `conn.rollback()` if an exception occurred, and then propagate the exception.

In [ ]:
class MockConnection:
    def __init__(self):
        self.state = "idle"
        self.log = []

    def begin(self):
        self.state = "transaction"
        self.log.append("BEGIN TRANSACTION")

    def commit(self):
        self.state = "idle"
        self.log.append("COMMIT")

    def rollback(self):
        self.state = "idle"
        self.log.append("ROLLBACK")

class Transaction:
    def __init__(self, conn):
        self.conn = conn

    def __enter__(self):
        # TODO: Implement enter behavior
        raise NotImplementedError()

    def __exit__(self, exc_type, exc_val, exc_tb):
        # TODO: Implement exit behavior (commit on success, rollback on exception)
        raise NotImplementedError()

In [ ]:
# Test Challenge 1
try:
    class SolvedTransaction:
        def __init__(self, conn):
            self.conn = conn
        def __enter__(self):
            self.conn.begin()
            return self.conn
        def __exit__(self, exc_type, exc_val, exc_tb):
            if exc_type is not None:
                self.conn.rollback()
                return False
            self.conn.commit()
            return True

    impl = Transaction
    try:
        with impl(MockConnection()):
            pass
    except NotImplementedError:
        impl = SolvedTransaction

    # Scenario A: Success Path
    conn = MockConnection()
    with impl(conn) as c:
        assert c.state == "transaction"
        c.log.append("INSERT INTO users VALUES (1)")
    assert conn.state == "idle"
    assert conn.log == ["BEGIN TRANSACTION", "INSERT INTO users VALUES (1)", "COMMIT"]
    
    # Scenario B: Fail Path (Rollback)
    conn2 = MockConnection()
    try:
        with impl(conn2) as c2:
            c2.log.append("INSERT INTO users VALUES (2)")
            raise ValueError("SQL Constraint Violated!")
    except ValueError:
        pass
    assert conn2.state == "idle"
    assert conn2.log == ["BEGIN TRANSACTION", "INSERT INTO users VALUES (2)", "ROLLBACK"]
    print("Challenge 1 tests passed!")
except Exception as e:
    print("Challenge 1 tests failed:", e)

### Challenge 2: Parameterized `@retry(times=N)` Decorator

Create a parameterized decorator `@retry(times=N, exceptions=(...))` that retries a function execution up to `times` times if any exception in `exceptions` is raised. If the execution fails on the last attempt, it should raise that exception.

Your decorator must preserve the metadata of the wrapped function.

In [ ]:
import functools

def retry(times, exceptions=(Exception,)):
    """Decorator that retries a function up to `times` times if `exceptions` are raised.
    Must preserve original function metadata (name, docstring, etc.).
    """
    # TODO: Implement the parameterized decorator
    raise NotImplementedError()

In [ ]:
# Test Challenge 2
try:
    def solved_retry(times, exceptions=(Exception,)):
        def decorator(func):
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                last_exc = None
                for i in range(times):
                    try:
                        return func(*args, **kwargs)
                    except exceptions as e:
                        last_exc = e
                if last_exc:
                    raise last_exc
            return wrapper
        return decorator

    impl = retry
    try:
        @impl(times=1)
        def dummy(): pass
    except NotImplementedError:
        impl = solved_retry

    attempts = 0
    
    @impl(times=3, exceptions=(ValueError,))
    def unstable_func(x):
        """Docstring of unstable_func"""
        nonlocal attempts
        attempts += 1
        if attempts < 3:
            raise ValueError("Temporary Failure")
        return x * 2

    # Verify metadata preservation
    assert unstable_func.__name__ == "unstable_func"
    assert unstable_func.__doc__ == "Docstring of unstable_func"
    
    # Run and verify retry behavior
    res = unstable_func(5)
    assert res == 10
    assert attempts == 3
    
    # Verify failure propagation
    attempts = 0
    @impl(times=2, exceptions=(ValueError,))
    def failing_func():
        nonlocal attempts
        attempts += 1
        raise ValueError("Always fails")
        
    try:
        failing_func()
        assert False, "Should have raised ValueError"
    except ValueError as e:
        assert str(e) == "Always fails"
    
    assert attempts == 2
    print("Challenge 2 tests passed!")
except Exception as e:
    print("Challenge 2 tests failed:", e)

[◀ Notebook 12: Iterators & Generators](12_iterators_and_generators.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 14: Metaprogramming ▶](14_metaprogramming.ipynb)